# K-Means Clustering: Pakistan COVID-19 (Complete Python Workflow)

This single notebook covers the whole Python part of the clustering guide (Parts 4 to 9). Run it from top to bottom (**Run All**). It replaces the earlier `KMeans_Clustering_Analysis` notebook.

| Section | Content |
|:--|:--|
| 1-3 | Setup, data loading and cleaning, aggregation (city, province, daily, weekly) |
| 4-6 | Feature scaling, optimal K (elbow, silhouette, Davies-Bouldin, Calinski-Harabasz), final K-Means and cluster profiles |
| 7 | Visualisations: 2D scatter, 3D scatter, PCA, radar, silhouette plot, interactive Plotly |
| 8 | Cluster quality metrics |
| 9-11 | Province, time-period (weekly) and travel-history clustering |
| 12-13 | Hierarchical clustering and the consensus choice of K |
| 14 | Export to Excel and CSV |
| 15-16 | Interpretation of the actual results and the 12-panel dashboard |
| 17 | Advanced techniques: DBSCAN, Gaussian mixtures, BIC |
| 18 | Extra checks: log transform, K-Means from scratch (matches the Excel sheets), stability |

**Where the files go.** Figures, CSVs and the Excel report are written to `outputs/clustering/` inside your project. Set `SAVE_OUTPUTS = False` in the first cell to only display results.

**Two things to know before reading the results**
1. The guide's feature set is Total_Cases, Total_Deaths, Total_Recovered, CFR and Recovery_Rate on raw counts. Karachi and Lahore are so much larger than every other city that raw counts mostly create a "giant cities" cluster and one huge cluster of everything else. Section 18 shows the same analysis with a `log(1 + x)` transform, which is what the Excel sheets use.
2. The text in Sections 15 and 19 is computed from your data, so it always matches the numbers printed above it.

In [ ]:
# ============================================================
# 1. SETUP AND IMPORTS
# ============================================================
import warnings
from collections import Counter
from math import pi
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score,
                             silhouette_samples, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)

SAVE_OUTPUTS = True          # False = only display, write nothing
K_OPTIMAL = 3
colors = ["#E63946", "#2A9D8F", "#F4A261"]
markers = ["o", "s", "^"]

# If the automatic search fails, put the full path of your csv here, e.g. r"D:\DA projects\Covid-19-portfolio\data\raw\PK COVID-19-3jun.csv"
CSV_PATH = None

def find_csv():
    if CSV_PATH:
        return Path(CSV_PATH)
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        found = [p for p in folder.rglob("*.csv")
                 if "covid" in p.name.lower() and not any(x in p.parts for x in ("outputs", "venv", ".venv", "site-packages"))
                 and not p.name.lower().startswith(("arima", "kmeans", "ml_", "city_", "province_", "travel_"))]
        if found:
            found.sort(key=lambda p: ("clean" not in p.name.lower(), len(str(p))))   # cleaned file first, else the raw one
            return found[0]
    raise FileNotFoundError("No COVID csv found. Set CSV_PATH in this cell to the full path of your data file.")

CSV = find_csv()
BASE = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
OUTD = BASE / "outputs" / "clustering"
if SAVE_OUTPUTS:
    OUTD.mkdir(parents=True, exist_ok=True)

def out(name):
    return OUTD / name

def save_fig(name):
    if SAVE_OUTPUTS:
        plt.savefig(out(name), dpi=150, bbox_inches="tight")

print("All libraries loaded | plotly available:", HAS_PLOTLY)
print("Data file :", CSV)
print("Outputs to:", OUTD if SAVE_OUTPUTS else "(not saving)")

## 2. Data Loading and Cleaning

In [ ]:
df = pd.read_csv(CSV)
df["Date"] = pd.to_datetime(df["Date"], format="mixed", errors="coerce")
df = df.dropna(subset=["Date"])

# Clean strings (blank travel history becomes "Unknown", as in the guide)
df["Travel_history"] = df["Travel_history"].fillna("Unknown").astype(str).str.strip()
for col in ["Province", "City"]:
    df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()

# Extra cleaning beyond the guide: misspelt city names and exact duplicate rows
CITY_FIXES = {"Bhakhar": "Bhakkar", "Dira Ismail Khan": "Dera Ismail Khan", "Jehlum": "Jhelum", "Kolai Pllas": "Kolai Pallas",
              "Kurrum": "Kurram", "Mirpur Khas": "Mirpurkhas", "Nankana": "Nankana Sahib", "Sahewal": "Sahiwal",
              "Shujawal": "Sujawal", "Tando Alahyar": "Tando Allahyar", "Torghar": "Tor Ghar"}
df["City"] = df["City"].replace(CITY_FIXES)
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df)} exact duplicate rows")

df_unclipped = df.copy()                       # kept for Section 18 (matches the Excel sheets, which keep negative corrections)

# Clean negatives (data errors): a few Recovered values are negative corrections
print("Negative values clipped to 0:", {c: int((df[c] < 0).sum()) for c in ["Cases", "Deaths", "Recovered"]})
for col in ["Cases", "Deaths", "Recovered"]:
    df[col] = df[col].clip(lower=0)

# Derived columns
df["Active"] = df["Cases"] - df["Deaths"] - df["Recovered"]
df["Day_Num"] = (df["Date"] - df["Date"].min()).dt.days + 1

print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")
print(f"Cities: {df['City'].nunique()}, Provinces: {df['Province'].nunique()}")

## 3. Aggregation Levels (City, Province, Daily, Weekly)

In [ ]:
def add_rates(frame):
    frame["CFR"] = np.where(frame["Total_Cases"] > 0, frame["Total_Deaths"] / frame["Total_Cases"] * 100, 0)
    frame["Recovery_Rate"] = np.where(frame["Total_Cases"] > 0, frame["Total_Recovered"] / frame["Total_Cases"] * 100, 0)
    return frame

# LEVEL 1: city
city_agg = df.groupby(["City", "Province"]).agg(
    Total_Cases=("Cases", "sum"), Total_Deaths=("Deaths", "sum"), Total_Recovered=("Recovered", "sum"),
    Active=("Active", "sum"), Records=("Cases", "count")).reset_index()
city_agg = add_rates(city_agg)
n_all_cities = len(city_agg)
city_agg = city_agg[city_agg["Total_Cases"] >= 5].reset_index(drop=True)      # cities with enough data
print(f"Cities with >= 5 cases: {len(city_agg)} of {n_all_cities}")
print(city_agg.sort_values("Total_Cases", ascending=False).head(10).to_string(index=False))

# LEVEL 2: province
province_agg = df.groupby("Province").agg(
    Total_Cases=("Cases", "sum"), Total_Deaths=("Deaths", "sum"), Total_Recovered=("Recovered", "sum"),
    Active=("Active", "sum"), Records=("Cases", "count")).reset_index()
province_agg = add_rates(province_agg)
print("\nProvince summary:")
print(province_agg.round(2).to_string(index=False))

# LEVEL 3: daily and weekly
daily = df.groupby("Date").agg(Cases=("Cases", "sum"), Deaths=("Deaths", "sum"), Recovered=("Recovered", "sum")).reset_index()
daily["Active"] = daily["Cases"] - daily["Deaths"] - daily["Recovered"]
daily["Day_Num"] = (daily["Date"] - daily["Date"].min()).dt.days + 1
daily["Week"] = daily["Date"].dt.isocalendar().week.astype(int)
weekly = daily.groupby("Week").agg(Cases=("Cases", "sum"), Deaths=("Deaths", "sum"), Recovered=("Recovered", "sum"),
                                   Active=("Active", "mean"), Days_in_Week=("Cases", "count")).reset_index()
print("\nWeekly aggregation shape:", weekly.shape)

## 4. Feature Selection and Scaling

In [ ]:
features = ["Total_Cases", "Total_Deaths", "Total_Recovered", "CFR", "Recovery_Rate"]
X_city = city_agg[features].copy()

scaler = StandardScaler()
X_city_scaled = scaler.fit_transform(X_city)
X_city_scaled_df = pd.DataFrame(X_city_scaled, columns=features)

print("Scaled feature statistics (mean 0, std 1):")
print(X_city_scaled_df.describe().round(2).to_string())

## 5. Optimal K: Elbow, Silhouette, Davies-Bouldin, Calinski-Harabasz

In [ ]:
wcss, sil_scores, db_scores, ch_scores = [], [], [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_city_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_city_scaled, labels))
    db_scores.append(davies_bouldin_score(X_city_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_city_scaled, labels))

results_k = pd.DataFrame({"K": list(K_range), "WCSS": wcss, "Silhouette": sil_scores,
                          "Davies_Bouldin": db_scores, "Calinski_Harabasz": ch_scores})
print(results_k.round(4).to_string(index=False))

best_sil_k = int(results_k.loc[results_k["Silhouette"].idxmax(), "K"])
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].plot(results_k["K"], results_k["WCSS"], "bo-", linewidth=2, markersize=10)
axes[0, 0].set(title="Elbow Method: WCSS", xlabel="K", ylabel="WCSS (Inertia)")
axes[0, 0].axvline(x=3, color="red", linestyle="--", alpha=0.5, label="K=3")
axes[0, 0].legend()
axes[0, 1].plot(results_k["K"], results_k["Silhouette"], "go-", linewidth=2, markersize=10)
axes[0, 1].set(title="Silhouette Score (Higher is Better)", xlabel="K", ylabel="Silhouette")
axes[0, 1].axvline(x=best_sil_k, color="red", linestyle="--", label=f"Best K={best_sil_k}")
axes[0, 1].legend()
axes[1, 0].plot(results_k["K"], results_k["Davies_Bouldin"], "ro-", linewidth=2, markersize=10)
axes[1, 0].set(title="Davies-Bouldin Index (Lower is Better)", xlabel="K", ylabel="Davies-Bouldin")
axes[1, 1].plot(results_k["K"], results_k["Calinski_Harabasz"], "mo-", linewidth=2, markersize=10)
axes[1, 1].set(title="Calinski-Harabasz Index (Higher is Better)", xlabel="K", ylabel="Calinski-Harabasz")
plt.tight_layout()
save_fig("clustering_optimal_k.png")
plt.show()

## 6. Apply K-Means with the Chosen K

Cluster numbers (0, 1, 2) are arbitrary. The `Cluster_Name` column ranks the clusters by average cases: Epicenter (highest), Hotspot, Low-Burden.

In [ ]:
kmeans_final = KMeans(n_clusters=K_OPTIMAL, init="k-means++", n_init=20, max_iter=300, random_state=42)
city_agg["Cluster"] = kmeans_final.fit_predict(X_city_scaled)

rank = city_agg.groupby("Cluster")["Total_Cases"].mean().rank(ascending=False).astype(int)
NAMES = {1: "Epicenter", 2: "Hotspot", 3: "Low-Burden"}
cluster_names = {c: NAMES[r] for c, r in rank.items()}
city_agg["Cluster_Name"] = city_agg["Cluster"].map(cluster_names)

print("Cluster distribution:")
print(city_agg.groupby(["Cluster", "Cluster_Name"]).size().to_string())

cluster_profile = city_agg.groupby("Cluster").agg(
    Cluster_Name=("Cluster_Name", "first"),
    Total_Cases_mean=("Total_Cases", "mean"), Total_Cases_sum=("Total_Cases", "sum"),
    Total_Deaths_mean=("Total_Deaths", "mean"), Total_Recovered_mean=("Total_Recovered", "mean"),
    CFR_mean=("CFR", "mean"), Recovery_Rate_mean=("Recovery_Rate", "mean"), N_Cities=("City", "count"))
print("\nCluster profiles:")
print(cluster_profile.round(2).to_string())

for c in range(K_OPTIMAL):
    print(f"\n--- Cluster {c} ({cluster_names[c]}), top 5 by cases ---")
    print(city_agg[city_agg["Cluster"] == c].nlargest(5, "Total_Cases")[["City", "Province", "Total_Cases", "Total_Deaths", "CFR"]].round(2).to_string(index=False))

## 7. Cluster Visualisation from Several Angles

In [ ]:
# Visualization 1: 2D scatter plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (x_col, y_col) in zip(axes, [("Total_Cases", "Total_Deaths"), ("Total_Cases", "CFR"), ("Total_Deaths", "Recovery_Rate")]):
    for c in range(K_OPTIMAL):
        subset = city_agg[city_agg["Cluster"] == c]
        ax.scatter(subset[x_col], subset[y_col], c=colors[c], marker=markers[c], s=80, alpha=0.7, edgecolors="black",
                   label=f"Cluster {c} {cluster_names[c]} (n={len(subset)})")
    ax.set(xlabel=x_col, ylabel=y_col, title=f"{x_col} vs {y_col}")
    ax.legend(fontsize=8)
plt.tight_layout()
save_fig("clustering_scatter_2d.png")
plt.show()

In [ ]:
# Visualization 2: 3D scatter
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
for c in range(K_OPTIMAL):
    subset = city_agg[city_agg["Cluster"] == c]
    ax.scatter(subset["Total_Cases"], subset["Total_Deaths"], subset["Total_Recovered"], c=colors[c], marker=markers[c],
               s=80, alpha=0.7, label=f"Cluster {c} {cluster_names[c]}")
ax.set(xlabel="Total Cases", ylabel="Total Deaths", zlabel="Total Recovered", title="3D Cluster Visualization")
ax.legend()
save_fig("clustering_scatter_3d.png")
plt.show()

In [ ]:
# Visualization 3: PCA projection (5 features -> 2 components)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_city_scaled)
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Cluster"] = city_agg["Cluster"].values
pca_df["City"] = city_agg["City"].values
pca_df["Cases"] = city_agg["Total_Cases"].values

fig, ax = plt.subplots(figsize=(14, 9))
for c in range(K_OPTIMAL):
    subset = pca_df[pca_df["Cluster"] == c]
    ax.scatter(subset["PC1"], subset["PC2"], c=colors[c], s=subset["Cases"] / 10 + 50, alpha=0.7, edgecolors="black",
               label=f"Cluster {c} {cluster_names[c]}")
for _, row in pca_df.nlargest(15, "Cases").iterrows():
    ax.annotate(row["City"], (row["PC1"], row["PC2"]), fontsize=9, alpha=0.8)
ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)", title="K-Means Clusters: PCA Projection")
ax.legend()
save_fig("clustering_pca.png")
plt.show()
print(f"Total variance explained: {pca.explained_variance_ratio_.sum() * 100:.2f}%")

In [ ]:
# Visualization 4: radar chart of cluster profiles
radar_features = features
radar_data = city_agg.groupby("Cluster")[radar_features].mean()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-10)

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = [n / len(radar_features) * 2 * pi for n in range(len(radar_features))]
angles += angles[:1]
for c in range(K_OPTIMAL):
    values = radar_norm.loc[c].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, color=colors[c], label=f"Cluster {c} {cluster_names[c]}")
    ax.fill(angles, values, alpha=0.15, color=colors[c])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=11)
ax.set_title("Cluster Profiles (Radar)", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
save_fig("clustering_radar.png")
plt.show()

In [ ]:
# Visualization 5: silhouette plot
silhouette_vals = silhouette_samples(X_city_scaled, city_agg["Cluster"])
avg_score = silhouette_score(X_city_scaled, city_agg["Cluster"])

fig, ax = plt.subplots(figsize=(12, 8))
y_lower = 10
for c in range(K_OPTIMAL):
    c_sil = np.sort(silhouette_vals[city_agg["Cluster"] == c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil, facecolor=colors[c], edgecolor=colors[c], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * len(c_sil), f"C{c}", fontsize=12)
    y_lower = y_upper + 10
ax.axvline(avg_score, color="red", linestyle="--", label=f"Avg = {avg_score:.3f}")
ax.set(title="Silhouette Plot", xlabel="Silhouette Coefficient", ylabel="Cluster")
ax.legend()
save_fig("clustering_silhouette.png")
plt.show()

In [ ]:
# Visualization 6: interactive 3D scatter (Plotly)
if HAS_PLOTLY:
    plot_df = city_agg.assign(Cluster_Label=city_agg["Cluster"].astype(str) + " " + city_agg["Cluster_Name"])
    fig = px.scatter_3d(plot_df, x="Total_Cases", y="Total_Deaths", z="Total_Recovered", color="Cluster_Label",
                        hover_name="City", hover_data=["Province", "CFR", "Recovery_Rate"],
                        title="Interactive 3D K-Means Clusters", color_discrete_sequence=colors)
    if SAVE_OUTPUTS:
        fig.write_html(out("clustering_3d_interactive.html"), include_plotlyjs="cdn")
    fig.show()
else:
    print("plotly is not installed in this environment: run 'pip install plotly' to enable the interactive chart.")

## 8. Cluster Quality Metrics

In [ ]:
sil = silhouette_score(X_city_scaled, city_agg["Cluster"])
db = davies_bouldin_score(X_city_scaled, city_agg["Cluster"])
ch = calinski_harabasz_score(X_city_scaled, city_agg["Cluster"])
inertia = kmeans_final.inertia_

print("=" * 60)
print("CLUSTERING QUALITY METRICS")
print("=" * 60)
print(f"Silhouette Score:       {sil:.4f}  (range -1 to 1, higher is better)")
print(f"Davies-Bouldin Index:   {db:.4f}  (lower is better)")
print(f"Calinski-Harabasz:      {ch:.2f}  (higher is better)")
print(f"Inertia (WCSS):         {inertia:.2f}")

## 9. Province-Level Clustering

In [ ]:
province_features = ["Total_Cases", "Total_Deaths", "Total_Recovered", "CFR", "Recovery_Rate"]
X_prov_scaled = StandardScaler().fit_transform(province_agg[province_features])

# Elbow for provinces (K must be smaller than the number of provinces)
k_prov = list(range(2, min(8, len(province_agg))))
wcss_prov = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_prov_scaled).inertia_ for k in k_prov]
plt.figure(figsize=(10, 5))
plt.plot(k_prov, wcss_prov, "bo-", linewidth=2, markersize=10)
plt.title("Elbow Method: Province Clustering")
plt.xlabel("K"); plt.ylabel("WCSS")
plt.axvline(x=3, color="red", linestyle="--", alpha=0.5)
save_fig("clustering_province_elbow.png")
plt.show()

km_prov = KMeans(n_clusters=3, n_init=20, random_state=42)
province_agg["Cluster_Prov"] = km_prov.fit_predict(X_prov_scaled)
print("Province clusters:")
print(province_agg[["Province", "Total_Cases", "CFR", "Cluster_Prov"]].sort_values("Cluster_Prov").round(2).to_string(index=False))

plt.figure(figsize=(14, 8))
for c in range(3):
    subset = province_agg[province_agg["Cluster_Prov"] == c]
    plt.scatter(subset["Total_Cases"], subset["CFR"], s=subset["Total_Deaths"] / 10 + 100, c=colors[c], marker=markers[c],
                alpha=0.7, edgecolors="black", label=f"Cluster {c}")
for _, row in province_agg.iterrows():
    plt.annotate(row["Province"], (row["Total_Cases"], row["CFR"]), xytext=(5, 5), textcoords="offset points", fontsize=10)
plt.xscale("log")
plt.xlabel("Total Cases (log scale)"); plt.ylabel("CFR (%)")
plt.title("Province Clustering by Case Load and CFR")
plt.legend()
save_fig("clustering_province_scatter.png")
plt.show()

## 10. Time-Period Clustering (Weekly Transmission Phases)

In [ ]:
weekly_features = ["Cases", "Deaths", "Recovered", "Active"]
X_week_scaled = StandardScaler().fit_transform(weekly[weekly_features])

wcss_week = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_week_scaled).inertia_ for k in range(2, 8)]
print("Weekly WCSS for K = 2..7:", [round(w, 2) for w in wcss_week])

km_week = KMeans(n_clusters=3, n_init=20, random_state=42)
weekly["Phase"] = km_week.fit_predict(X_week_scaled)
phase_rank = weekly.groupby("Phase")["Cases"].mean().rank(ascending=True).astype(int)       # 1 = lowest activity
PHASE_NAMES = {1: "Early / low", 2: "Growth", 3: "Peak"}
weekly["Phase_Name"] = weekly["Phase"].map({p: PHASE_NAMES[r] for p, r in phase_rank.items()})

print("\nWeekly phases:")
print(weekly[["Week", "Cases", "Deaths", "Active", "Phase", "Phase_Name"]].round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(16, 7))
for c in range(3):
    subset = weekly[weekly["Phase"] == c]
    ax.scatter(subset["Week"], subset["Cases"], c=colors[c], s=150, marker=markers[c], alpha=0.8, edgecolors="black",
               label=f"Phase {c} ({subset['Phase_Name'].iloc[0] if len(subset) else ''})")
ax.plot(weekly["Week"], weekly["Cases"], "k--", alpha=0.3)
ax.set(xlabel="Week Number (ISO)", ylabel="Weekly Cases", title="K-Means Time-Period Clustering: Transmission Phases")
ax.legend()
save_fig("clustering_temporal.png")
plt.show()

## 11. Travel-History Clustering

In [ ]:
travel_agg = df.groupby("Travel_history").agg(
    Total_Cases=("Cases", "sum"), Total_Deaths=("Deaths", "sum"), Records=("Cases", "count"),
    Avg_Cases_Per_Record=("Cases", "mean")).reset_index()
travel_agg["CFR"] = np.where(travel_agg["Total_Cases"] > 0, travel_agg["Total_Deaths"] / travel_agg["Total_Cases"] * 100, 0)

n_travel_all = len(travel_agg)
travel_agg = travel_agg[travel_agg["Records"] >= 3].reset_index(drop=True)          # keep categories with at least 3 records
print(f"Travel categories with >= 3 records: {len(travel_agg)} of {n_travel_all}")

travel_features = ["Total_Cases", "Total_Deaths", "Records", "Avg_Cases_Per_Record", "CFR"]
X_travel_scaled = StandardScaler().fit_transform(travel_agg[travel_features])
km_travel = KMeans(n_clusters=3, n_init=20, random_state=42)
travel_agg["Cluster_Travel"] = km_travel.fit_predict(X_travel_scaled)

print("\nTravel history clusters:")
print(travel_agg[["Travel_history", "Total_Cases", "Records", "CFR", "Cluster_Travel"]].sort_values("Cluster_Travel").round(2).to_string(index=False))

## 12. Hierarchical Clustering (Alternative to K-Means)

In [ ]:
link = linkage(X_city_scaled, method="ward")
cut_height = (link[-3, 2] + link[-2, 2]) / 2          # height that leaves exactly 3 clusters

fig, ax = plt.subplots(figsize=(18, 8))
dendrogram(link, labels=city_agg["City"].values, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set(title="Hierarchical Clustering Dendrogram (Ward Linkage)", xlabel="City", ylabel="Distance")
ax.axhline(y=cut_height, color="red", linestyle="--", label=f"Cut at K=3 (height {cut_height:.1f})")
ax.legend()
plt.tight_layout()
save_fig("clustering_dendrogram.png")
plt.show()

agg_model = AgglomerativeClustering(n_clusters=3, linkage="ward")
city_agg["Cluster_Hier"] = agg_model.fit_predict(X_city_scaled)
agreement = pd.crosstab(city_agg["Cluster"], city_agg["Cluster_Hier"])
print("Agreement between K-Means (rows) and Hierarchical (columns):")
print(agreement.to_string())
print("Adjusted Rand index:", round(adjusted_rand_score(city_agg["Cluster"], city_agg["Cluster_Hier"]), 3))

## 13. Optimal K: Consensus of Several Metrics

In [ ]:
print("=" * 70)
print("OPTIMAL K SELECTION: MULTIPLE METHODS")
print("=" * 70)
print(results_k.round(4).to_string(index=False))

best_k_sil = int(results_k.loc[results_k["Silhouette"].idxmax(), "K"])
best_k_db = int(results_k.loc[results_k["Davies_Bouldin"].idxmin(), "K"])
best_k_ch = int(results_k.loc[results_k["Calinski_Harabasz"].idxmax(), "K"])
print(f"\nBest K by Silhouette:        {best_k_sil}")
print(f"Best K by Davies-Bouldin:    {best_k_db}")
print(f"Best K by Calinski-Harabasz: {best_k_ch}")

votes = Counter([best_k_sil, best_k_db, best_k_ch])
consensus_k, n_votes = votes.most_common(1)[0]
print(f"\nConsensus K: {consensus_k} ({n_votes} of 3 metrics)" if n_votes > 1 else
      f"\nNo majority: the three metrics disagree {sorted(votes)}. Choose K from the elbow plot and the interpretation, not from one metric.")
print(f"K used for the final model: {K_OPTIMAL}")

## 14. Export Results

In [ ]:
if SAVE_OUTPUTS:
    centroids_df = pd.DataFrame(kmeans_final.cluster_centers_, columns=features,
                                index=[f"Cluster_{i}" for i in range(K_OPTIMAL)])
    with pd.ExcelWriter(out("Clustering_Report.xlsx"), engine="openpyxl") as writer:
        df.assign(Date=df["Date"].dt.date).to_excel(writer, sheet_name="Clean_Data", index=False)
        city_agg.to_excel(writer, sheet_name="City_Aggregates", index=False)
        province_agg.to_excel(writer, sheet_name="Province_Aggregates", index=False)
        travel_agg.to_excel(writer, sheet_name="Travel_Aggregates", index=False)
        daily.assign(Date=daily["Date"].dt.date).to_excel(writer, sheet_name="Daily", index=False)
        weekly.to_excel(writer, sheet_name="Weekly", index=False)
        city_agg[["City", "Province", "Total_Cases", "Total_Deaths", "Total_Recovered", "CFR", "Cluster", "Cluster_Name"]].to_excel(
            writer, sheet_name="City_Clusters", index=False)
        province_agg[["Province", "Total_Cases", "Total_Deaths", "CFR", "Cluster_Prov"]].to_excel(writer, sheet_name="Province_Clusters", index=False)
        travel_agg[["Travel_history", "Total_Cases", "Records", "CFR", "Cluster_Travel"]].to_excel(writer, sheet_name="Travel_Clusters", index=False)
        results_k.to_excel(writer, sheet_name="Optimal_K", index=False)
        centroids_df.to_excel(writer, sheet_name="Centroids")
        cluster_profile.to_excel(writer, sheet_name="Cluster_Profiles")
    city_agg[["City", "Province", "Cluster", "Cluster_Name"]].to_csv(out("city_clusters.csv"), index=False)
    province_agg[["Province", "Cluster_Prov"]].to_csv(out("province_clusters.csv"), index=False)
    print("Exported:", out("Clustering_Report.xlsx").name, "| city_clusters.csv | province_clusters.csv")
    print("Folder  :", OUTD)
else:
    print("SAVE_OUTPUTS is False: nothing exported.")

## 15. Interpreting the Results (computed from your data)

In [ ]:
print("=" * 70)
print("CITY CLUSTERS (K = 3)")
print("=" * 70)
actions = {"Epicenter": "maximum hospital capacity, strictest measures",
           "Hotspot": "targeted testing and contact tracing",
           "Low-Burden": "awareness campaigns, prevent growth into hotspots"}
total_cases = city_agg["Total_Cases"].sum()
for c in sorted(cluster_names, key=lambda x: {"Epicenter": 0, "Hotspot": 1, "Low-Burden": 2}[cluster_names[x]]):
    g = city_agg[city_agg["Cluster"] == c]
    top = ", ".join(g.nlargest(5, "Total_Cases")["City"])
    print(f"\nCluster {c} - {cluster_names[c]}: {len(g)} cities, {g['Total_Cases'].sum() / total_cases:.0%} of cases")
    print(f"  average {g['Total_Cases'].mean():,.0f} cases per city | CFR {g['Total_Deaths'].sum() / max(g['Total_Cases'].sum(), 1) * 100:.2f}% | top: {top}")
    print(f"  suggested focus: {actions[cluster_names[c]]}")

print("\n" + "=" * 70)
print("WEEKLY PHASES")
print("=" * 70)
for name in ["Early / low", "Growth", "Peak"]:
    g = weekly[weekly["Phase_Name"] == name]
    if len(g):
        print(f"{name:<12}: weeks {', '.join(map(str, g['Week']))} | average {g['Cases'].mean():,.0f} cases per week")
print("Note: the first week (26 Feb to 1 Mar) and the last week (1 to 3 Jun) are partial weeks, so their totals are lower; see Days_in_Week.")

print("\n" + "=" * 70)
print("PROVINCES AND TRAVEL HISTORY")
print("=" * 70)
for c in range(3):
    print(f"Province cluster {c}:", ", ".join(province_agg.loc[province_agg["Cluster_Prov"] == c, "Province"]))
for c in range(3):
    print(f"Travel cluster {c}:", ", ".join(travel_agg.loc[travel_agg["Cluster_Travel"] == c, "Travel_history"]))

## 16. Complete Visual Dashboard

In [ ]:
fig = plt.figure(figsize=(22, 18))
gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(results_k["K"], results_k["WCSS"], "bo-", linewidth=2)
ax1.axvline(x=K_OPTIMAL, color="red", linestyle="--")
ax1.set(title="Elbow Method", xlabel="K", ylabel="WCSS")

ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(results_k["K"], results_k["Silhouette"], color="steelblue", edgecolor="black")
ax2.axhline(y=sil, color="red", linestyle="--", label=f"Chosen K={K_OPTIMAL}")
ax2.set(title="Silhouette Scores", xlabel="K", ylabel="Silhouette")
ax2.legend()

ax3 = fig.add_subplot(gs[0, 2])
sizes = city_agg["Cluster"].value_counts().sort_index()
ax3.bar(sizes.index, sizes.values, color=colors, edgecolor="black")
ax3.set(title="Cluster Sizes", xlabel="Cluster", ylabel="Number of Cities")
ax3.set_xticks(sizes.index)

ax4 = fig.add_subplot(gs[1, 0])
for c in range(K_OPTIMAL):
    s = city_agg[city_agg["Cluster"] == c]
    ax4.scatter(s["Total_Cases"].clip(lower=0.5), s["Total_Deaths"].clip(lower=0.5), c=colors[c], s=80, alpha=0.7, edgecolors="black", label=f"C{c}")
ax4.set_xscale("log"); ax4.set_yscale("log")
ax4.set(title="Cases vs Deaths (log scales; zeros shown at 0.5)", xlabel="Total Cases", ylabel="Total Deaths")
ax4.legend()

ax5 = fig.add_subplot(gs[1, 1])
for c in range(K_OPTIMAL):
    s = city_agg[city_agg["Cluster"] == c]
    ax5.scatter(s["Total_Cases"], s["CFR"], c=colors[c], s=80, alpha=0.7, edgecolors="black", label=f"C{c}")
ax5.set_xscale("log")
ax5.set(title="Cases vs CFR", xlabel="Total Cases", ylabel="CFR (%)")
ax5.legend()

ax6 = fig.add_subplot(gs[1, 2], projection="3d")
for c in range(K_OPTIMAL):
    s = city_agg[city_agg["Cluster"] == c]
    ax6.scatter(s["Total_Cases"], s["Total_Deaths"], s["Total_Recovered"], c=colors[c], s=60, alpha=0.7)
ax6.set(title="3D View", xlabel="Cases", ylabel="Deaths", zlabel="Recov.")

ax7 = fig.add_subplot(gs[2, 0])
for c in range(K_OPTIMAL):
    s = pca_df[pca_df["Cluster"] == c]
    ax7.scatter(s["PC1"], s["PC2"], c=colors[c], alpha=0.7, edgecolors="black")
ax7.set(title="PCA Projection", xlabel="PC1", ylabel="PC2")

ax8 = fig.add_subplot(gs[2, 1])
y_lower = 10
for c in range(K_OPTIMAL):
    c_sil = np.sort(silhouette_vals[city_agg["Cluster"] == c])
    y_upper = y_lower + len(c_sil)
    ax8.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil, facecolor=colors[c], edgecolor=colors[c], alpha=0.7)
    ax8.text(-0.05, y_lower + 0.5 * len(c_sil), str(c), fontsize=10)
    y_lower = y_upper + 10
ax8.axvline(sil, color="red", linestyle="--")
ax8.set_title("Silhouette Plot")

ax9 = fig.add_subplot(gs[2, 2])
heat = city_agg.groupby("Cluster")[features].mean()
heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-10)
sns.heatmap(heat_norm, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax9)
ax9.set_title("Cluster Profiles (Normalized)")

ax10 = fig.add_subplot(gs[3, 0])
for c in range(3):
    s = weekly[weekly["Phase"] == c]
    ax10.scatter(s["Week"], s["Cases"], c=colors[c], s=150, alpha=0.8, edgecolors="black", label=f"Phase {c}")
ax10.plot(weekly["Week"], weekly["Cases"], "k--", alpha=0.3)
ax10.set(title="Temporal Phases", xlabel="Week", ylabel="Cases")
ax10.legend()

ax11 = fig.add_subplot(gs[3, 1])
for c in range(3):
    s = province_agg[province_agg["Cluster_Prov"] == c]
    ax11.scatter(s["Total_Cases"], s["CFR"], s=200, c=colors[c], alpha=0.7, edgecolors="black", label=f"C{c}")
for _, row in province_agg.iterrows():
    ax11.annotate(row["Province"][:8], (row["Total_Cases"], row["CFR"]), fontsize=8)
ax11.set_xscale("log")
ax11.set(title="Province Clusters", xlabel="Cases", ylabel="CFR")
ax11.legend()

ax12 = fig.add_subplot(gs[3, 2])
for c in range(3):
    s = travel_agg[travel_agg["Cluster_Travel"] == c]
    ax12.scatter(s["Records"], s["Total_Cases"], s=200, c=colors[c], alpha=0.7, edgecolors="black", label=f"C{c}")
for _, row in travel_agg.iterrows():
    ax12.annotate(row["Travel_history"][:12], (row["Records"], row["Total_Cases"]), fontsize=8)
ax12.set_yscale("log")
ax12.set(title="Travel History Clusters", xlabel="Records", ylabel="Total Cases")
ax12.legend()

plt.suptitle("K-MEANS CLUSTERING DASHBOARD: PAKISTAN COVID-19", fontsize=20, fontweight="bold", y=0.995)
save_fig("clustering_dashboard.png")
plt.show()

## 17. Advanced Clustering Techniques

In [ ]:
# 17.1 DBSCAN (density based): label -1 = outlier / noise
dbscan = DBSCAN(eps=1.5, min_samples=3)
city_agg["DBSCAN"] = dbscan.fit_predict(X_city_scaled)
print("DBSCAN clusters (-1 = noise):")
print(city_agg["DBSCAN"].value_counts().sort_index().to_string())
print("Cities marked as noise:", ", ".join(city_agg.loc[city_agg["DBSCAN"] == -1, "City"]) or "none")

In [ ]:
# 17.2 Gaussian Mixture Model (soft clustering)
gmm = GaussianMixture(n_components=3, random_state=42)
city_agg["GMM_Cluster"] = gmm.fit_predict(X_city_scaled)
probs = gmm.predict_proba(X_city_scaled)
print("GMM cluster sizes:", np.bincount(city_agg["GMM_Cluster"]).tolist())
print("\nGMM cluster probabilities (first 5 cities):")
print(pd.concat([city_agg[["City"]].head(), pd.DataFrame(probs, columns=["P_C0", "P_C1", "P_C2"]).head()], axis=1).round(3).to_string(index=False))

In [ ]:
# 17.3 Optimal number of components for the GMM via BIC / AIC
ks = list(range(2, 8))
bic_scores, aic_scores = [], []
for k in ks:
    g = GaussianMixture(n_components=k, random_state=42).fit(X_city_scaled)
    bic_scores.append(g.bic(X_city_scaled)); aic_scores.append(g.aic(X_city_scaled))
plt.figure(figsize=(12, 5))
plt.plot(ks, bic_scores, "bo-", label="BIC")
plt.plot(ks, aic_scores, "ro-", label="AIC")
plt.title("GMM Model Selection (Lower is Better)")
plt.xlabel("K"); plt.legend()
save_fig("clustering_gmm_bic.png")
plt.show()
print("Lowest BIC at K =", ks[int(np.argmin(bic_scores))], "| lowest AIC at K =", ks[int(np.argmin(aic_scores))])

## 18. Extra Checks: Log Transform, K-Means from Scratch, Stability

This section matches the Excel workbook (`KMeans_City` and related sheets): three features (Cases, Deaths, Recovered), all 117 cities, `log(1 + x)` then z-scores, and farthest-first starting centroids.

In [ ]:
FEATS3 = ["Cases", "Deaths", "Recovered"]
cities3 = df_unclipped.groupby("City")[FEATS3].sum().sort_values("Cases", ascending=False)

def prepare(frame, log=True):
    x = frame[FEATS3].clip(lower=0).astype(float)
    if log:
        x = np.log1p(x)
    return ((x - x.mean()) / x.std(ddof=1)).to_numpy()

def farthest_first(Zm, k):
    idx = [0]
    while len(idx) < k:
        d = np.min(((Zm[:, None, :] - Zm[idx][None]) ** 2).sum(2), axis=1)
        idx.append(int(d.argmax()))
    return idx

def run_kmeans(Zm, k):
    return KMeans(n_clusters=k, init=Zm[farthest_first(Zm, k)], n_init=1, max_iter=300, random_state=42).fit(Zm)

Z_log, Z_raw = prepare(cities3), prepare(cities3, log=False)
km_log, km_raw = run_kmeans(Z_log, 3), run_kmeans(Z_raw, 3)
print(f"{len(cities3)} cities (all of them, no minimum-cases filter)")
print("Cluster sizes with raw counts :", np.bincount(km_raw.labels_).tolist(), "| mean silhouette", round(silhouette_score(Z_raw, km_raw.labels_), 3))
print("Cluster sizes with log(1 + x) :", np.bincount(km_log.labels_).tolist(), "| mean silhouette", round(silhouette_score(Z_log, km_log.labels_), 3))
for c in range(3):
    members = cities3.index[km_raw.labels_ == c]
    if len(members) <= 3:
        print(f"  raw-count cluster with only {len(members)} city/cities: {', '.join(members)}")
print("Agreement between raw and log solutions (adjusted Rand index):", round(adjusted_rand_score(km_raw.labels_, km_log.labels_), 3))
print(f"\nLog-transformed solution: WCSS {km_log.inertia_:.4f} (the Excel sheet KMeans_City shows the same value when the log switch is 1)")

In [ ]:
def kmeans_scratch(Zm, start_rows, max_iter=50):
    centroids = Zm[start_rows].copy()
    labels = None
    for it in range(1, max_iter + 1):
        dist = np.sqrt(((Zm[:, None, :] - centroids[None]) ** 2).sum(axis=2))      # distance to each centroid
        new_labels = dist.argmin(axis=1)                                            # assign to the nearest centroid
        changed = len(Zm) if labels is None else int((new_labels != labels).sum())
        print(f"  iteration {it}: {changed} points reassigned")
        if labels is not None and changed == 0:
            break
        labels = new_labels
        centroids = np.array([Zm[labels == k].mean(axis=0) if (labels == k).any() else centroids[k] for k in range(len(centroids))])
    return labels, centroids, sum(((Zm[labels == k] - centroids[k]) ** 2).sum() for k in range(len(centroids)))

start = farthest_first(Z_log, 3)
print("Starting centroids:", list(cities3.index[start]))
labels_s, cent_s, wcss_s = kmeans_scratch(Z_log, start)
print(f"\nFrom scratch WCSS {wcss_s:.4f} | scikit-learn WCSS {km_log.inertia_:.4f}")
print("Identical cluster assignments:", bool((labels_s == km_log.labels_).all()))

In [ ]:
reference = km_log.labels_
scores = [adjusted_rand_score(reference, KMeans(n_clusters=3, init="random", n_init=1, random_state=s).fit(Z_log).labels_) for s in range(20)]
print("Stability over 20 random starts (adjusted Rand index vs the farthest-first solution)")
print(f"  min {min(scores):.3f} | mean {np.mean(scores):.3f} | max {max(scores):.3f}")
print("1.0 = same grouping. Lower values mean a few cities move between neighbouring tiers depending on the start.")

## 19. Summary and Comparison with Excel

| Aspect | Excel | Python (scikit-learn) |
|:--|:--|:--|
| Built-in K-Means | No (add-in or formulas) | Yes |
| Manual implementation | Yes (the KMeans_ sheets use formulas) | Yes (Section 18) |
| Standardisation | STANDARDIZE or (x - AVERAGE) / STDEV | StandardScaler |
| Elbow method | Repeat per K and chart (Elbow_Silhouette) | Loop over K |
| Silhouette | Formula-based (Distance_Matrix) | Built in |
| 3D plot, PCA, radar | No | Yes |
| Hierarchical, DBSCAN, GMM | No | Yes |
| Interactive charts | No | Yes (Plotly) |

**Points to state when presenting the results**
- Cluster numbers are arbitrary. The names (Epicenter, Hotspot, Low-Burden) come from average cases.
- The guide's raw-count features let a handful of very large cities dominate; Section 18 shows the log-transformed version used in Excel.
- The choice of K is a judgement. When the metrics disagree, the output of Section 13 says so; explain the choice using the elbow plot and the interpretation.
- Province (8 rows) and travel-history (a few rows) clusters are small samples: read them as tiers, not as findings.
- Outputs are in `outputs/clustering/`: PNG figures, `Clustering_Report.xlsx`, `city_clusters.csv`, `province_clusters.csv` and the interactive HTML (when Plotly is installed).